# Distinguishing Non-orthogonal Quantum States Workbook

This workbook describes the solutions to the problems offered in the "Distinguishing Non-orthogonal Quantum States" kata. Since the tasks are offered as programming problems, the explanations also cover some elements of Workbench that might be non-obvious for a first-time user.

In [ ]:
from psiqdk.workbench import Qubits


## Problem 1. Distinguish $\ket{0}$ and $\ket{+}$

Let ${\ket{E_a}, \ket{E_b}}$ be a measurement with two outcomes $a$ and $b$, which we identify with the answers, i.e., outcome "a" means we answer "state was $\ket{0}$" and outcome "b" means we answer "state was $\ket{+}$". Then we define

* $P(a|0)$ = probability to observe first outcome given that the state was $\ket{0}$
* $P(b|0)$ = probability to observe second outcome given that the state was $\ket{0}$
* $P(a|+)$ = probability to observe first outcome given that the state was $\ket{+}$
* $P(b|+)$ = probability to observe second outcome given that the state was $\ket{+}$

The task is to maximize the probability to be correct on a single shot experiment, which is the same as to minimize the probability to be wrong on a single shot.

Since the task promises uniform prior distribution of the inputs $\ket{0}$ and $\ket{+}$, i.e., $P(+) = P(0) = \frac{1}{2}$, we get the following expression for the probability of giving a correct answer:

$$P_{correct} = P(0) P(a|0) + P(+) P(b|+) = \frac{1}{2} (P(a|0) + P(b|+))$$

We can represent our measurement as a von Neumann measurement of the following form:

$$\ket{E_a} = R_y(2\alpha) \begin{bmatrix} 1 \\ 0 \end{bmatrix} = \begin{bmatrix} \cos \alpha \\ \sin \alpha \end{bmatrix}$$
$$\ket{E_b} = R_y(2\alpha) \begin{bmatrix} 0 \\ 1 \end{bmatrix} = \begin{bmatrix} - \sin \alpha \\ \cos \alpha \end{bmatrix}$$

<center><img src="./media/task1_rotation.png" width="80%"/></center>

Using this representation, we can express our probabilities as follows:

$$P(a|0) = |\braket{E_a|0}|^2 = \cos^2 \alpha$$
    
$$P(b|+) = |\braket{E_b|+}|^2 = \frac{1}{2} - \cos \alpha \sin \alpha$$
    
$$P_{correct} = \frac{1}{2} (\cos^2 \alpha + \frac{1}{2} - \cos \alpha \sin \alpha)$$
    
Maximizing this for $\alpha$, we get max $P_{success} = \frac{1}{2} (1 + \frac{1}{\sqrt{2}}) = 0.8535...$, which is attained for $\alpha = -\pi/8$.
    
This means that $\ket{E_a}$ and $\ket{E_b}$ are the result of rotating $\ket{0}$ and $\ket{1}$, respectively, by $-\pi/8$. If we rotate the whole system by $-\alpha = \pi/8$, we will get $\ket{E_a}=\ket{0}$ and $\ket{E_b}=\ket{1}$, and a measurement in the computational basis will give us the correct result with a probability of 85%.

> Remember to multiply the rotation angle by $2$ when passing it to the `ry` method!

In [ ]:
def measure_zero_or_plus(reg: Qubits) -> int:
    reg.ry((1, 4))
    return reg.read()

## Problem 2. Distinguish $\ket{0}$, $\ket{+}$, or inconclusive ?

A simple strategy that gives an inconclusive result with probability 0.75 and never errs in case it yields a conclusive result can be obtained from randomizing the choice of measurement basis between the computational basis and the Hadamard basis.
    
Notice that when measured in the standard basis, the state $\ket{0}$ will always lead to the outcome "0", and the state $\ket{+}$ will lead to outcomes "0" and "1" with probability $\frac12$ each. This means that if we measure "1", we can with certainty conclude that the state was $\ket{+}$.
    
A similar argument applies to the scenario where we measure in the Hadamard basis, where $\ket{0}$ can lead to both "+" and "-" outcomes, and $\ket{+}$ always leads to "+". Then if we measured "-", we can with certainty conclude that the state was $\ket{0}$.
    
This leads to the following scenarios (shown are the conditional probabilities of the resulting answers in each of the above scenarios).

| State | Basis | P(0) | P(1) | P(-1) |
| --- | --- | --- | --- | --- |
| $\ket{0}$ | Computational | 0 | 0 | 1 |
| $\ket{+}$ | Computational | 0 | 0.5 | 0.5 |
| $\ket{0}$ | Hadamard | 0.5 | 0 | 0.5 |
| $\ket{+}$ | Hadamard | 0 | 0 | 1 |

Since each of the four scenarios occurs with probability 25%, overall this strategy ends up correctly identifying $\ket{0}$ and $\ket{+}$ states with 12.5% probability each and giving inconclusive result with 75% probability.

In [ ]:
from random import randint

def measure_zero_or_plus_or_inconclusive(reg: Qubits) -> int:
    if randint(0, 1):
        reg.had()
        result = reg.read()
        if result:
            return 0
        else:
            return -1
    
    else:
        result = reg.read()
        if result:
            return 1
        else:
            return -1

## Problem 3. Peres/Wooters game

> The task is a game inspired by a quantum detection problem due to Holevo ("Information-theoretical aspects of quantum measurement", A. Holevo) and Peres/Wootters ("Optimal detection of quantum information", A. Peres and W. K. Wootters). In the game, player A thinks of a number (0, 1 or 2) and the opponent, player B, tries to guess any number but the one chosen by player A. 
>
> Classically, if you just made a guess, you'd have to ask two questions to be right $100\%$ of the time. If instead, player A prepares a qubit with 0, 1, or 2 encoded into three single qubit states that are at an angle of 120 degrees with respect to each other and then hands the state to the opponent, then player B can apply a Positive Operator Valued Measure (POVM) consisting of 3 states that are perpendicular to the states chosen by player A. 
> It can be shown that this allows B to be right $100\%$ of the time with only 1 measurement, which is something that is not achievable with a von Neumann measurement on 1 qubit.
See also ("Quantum Theory: Concepts and Methods", A. Peres) for a nice description of the optimal POVM.
    
Next, we address how we can implement the mentioned POVM by way of a von Neumann measurement, and then how to implement said von Neumann measurement in Workbench. First, we note that the POVM elements are given by the columns of the following matrix (just simply use $\ket{A}, \ket{B}, \ket{C}$ as columns): 
     
$$M = \frac{1}{\sqrt{2}}\left(\begin{array}{rrr}
1 & 1 & 1 \\ 
1 & \omega & \omega^2 
\end{array}
\right)$$
    
where $\omega = e^{2 \pi i/3}$ denotes a primitive third root of unity. Our task will be to implement the rank 1 POVM given by the columns of $M$ via a von Neumann measurement. This can be done by \"embedding\" $M$ into a larger unitary matrix (taking complex conjugate and transpose):
    
$$M' = \frac{1}{\sqrt{3}}\left(\begin{array}{cccc}
1 & -1 & 1 & 0 \\ 
1 & -\omega^2 & \omega & 0 \\
1 & -\omega & \omega^2 & 0 \\
0 & 0 & 0 & -i\sqrt{3} 
\end{array}
\right)$$
    
If we apply a measurement defined by $M'$ to an input state given by column $k$ of $M$ (padded with two zeros to make it a vector of length $4$, e.g. $\ket{A} \otimes \ket{0}$ for $\ket{A}$ as the input) for $k \in \{1, 2, 3\}$, we will never get the label $k$ as a measurement result, because row $k$ of $M'$ is perpendicular to column $k$ of $M$.

We are therefore left with the problem of implementing $M'$ as a sequence of elementary quantum gates. 

Notice that 
    
$$M' \cdot {\rm diag}(1,-1,1,-1) = M' \cdot (\mathbf{1}_2 \otimes Z) = 
\frac{1}{\sqrt{3}}\left(\begin{array}{cccc}
1 & 1 & 1 & 0 \\ 
1 & \omega^2 & \omega & 0 \\
1 & \omega & \omega^2 & 0 \\
0 & 0 & 0 & i\sqrt{3} 
\end{array}
\right)$$
    
Using a technique used in the Rader (also sometimes called Rader-Winograd) decomposition of the discrete Fourier transform ("Discrete Fourier transforms when the number of data samples is prime", C. M. Rader), which reduces it to a cyclic convolution, we apply a $2\times 2$ Fourier transform on the indices $i,j=1,2$ of this matrix (i.e. a block matrix which consists of a direct sum of blocks $\mathbf{1}_1$, $H$, and $\mathbf{1}_1$ which we abbreviate in short as ${\rm diag}(1,H,1)$). 
    
This yields
    
$${\rm diag}(1, H, 1) \cdot M' \cdot (\mathbf{1}_2 \otimes Z) \cdot {\rm diag}(1, H, 1) = 
\left(\begin{array}{rrrr}
\frac{1}{\sqrt3} & \sqrt{\frac23} & 0 & 0 \\ 
\sqrt{\frac23} & -\frac{1}{\sqrt{3}} & 0 & 0 \\
0 & 0 & -i & 0 \\
0 & 0 & 0 & i 
\end{array}
\right)$$

> The transformation given by ${\rm diag}(1,H,1)$ is equivalent to $CNOT_{1,0}CH_{0,1}CNOT_{1,0}$.

This implies that after multiplication with the diagonal operator $(S^\dagger \otimes \mathbf{1}_2)$, we are left with 
    
$${\rm diag}(1, H, 1) \cdot M' \cdot (\mathbf{1}_2 \otimes Z) \cdot {\rm diag}(1, H, 1)\cdot (S^\dagger \otimes \mathbf{1}_2) = 
\left(\begin{array}{rrrr}
\frac{1}{\sqrt3} & \sqrt{\frac23} & 0 & 0 \\ 
\sqrt{\frac23} & -\frac{1}{\sqrt{3}} & 0 & 0 \\
0 & 0 & -1 & 0 \\
0 & 0 & 0 & 1 
\end{array}
\right) = \\
= \ket{0}\bra{0}\otimes \left( R_Y(2\arccos\sqrt{\frac13}) Z \right) + \ket{1}\bra{1}\otimes -Z = \\
= \left(\ket{0}\bra{0}\otimes -R_Y(2\arccos\sqrt{\frac13})\right) \cdot \left(\mathbf{1}_2\otimes -Z \right) = CRy \cdot \left(\mathbf{1}_2\otimes -Z \right)$$
    
which is a zero-controlled rotation $R$ around the $Y$-axis by an angle given by $\arccos \sqrt{\frac13}$ (plus a relative phase change on a basis state). 
    
Putting everything together, we can implement the matrix $M'$ by applying the inverses of gates:
    
$$M' = {\rm diag}(1,H,1) \cdot CRy \cdot (\mathbf{1}_2\otimes -Z) \cdot (S \otimes \mathbf{1}_2) \cdot {\rm diag}(1,H,1) \cdot (\mathbf{1}_2 \otimes Z)$$
    
Noting finally, that to apply this sequence of unitaries to a state, we have to apply it in reverse when writing it as a program (as actions on vectors are left-associative).

After the transformation, we simply measure both the auxiliary qubit and the register. This is now guaranteed to not collapse to the incorrect results.

In [ ]:
from psiqdk.workbench import units
from math import acos, sqrt

def peres_wooters_game(reg: Qubits) -> int:
    aux = Qubits(1, 'aux', reg.qpu)

    # I ⊗ Z
    reg.z()
    
    # diag(1, H, 1)
    # reg.x(cond=aux)     # We should apply this gate but it doesn't have effect when freshly allocated aux is 0
    aux.had(cond=reg)
    reg.x(cond=aux)

    # S ⊗ I
    aux.s()

    # I ⊗ -Z
    (reg|aux == 2).reflect()

    # CRy
    alpha = acos(sqrt(1/3))
    reg.ry(-2 * alpha * units.rad, cond=~aux)

    # diag(1, H, 1)
    reg.x(cond=aux)
    aux.had(cond=reg)
    reg.x(cond=aux)

    result_aux = aux.read()
    aux.release()

    result_reg = reg.read()
    if result_aux == 0 and result_reg == 0:
        return 0
    elif result_aux == 1 and result_reg == 0:
        return 1
    elif result_aux == 0 and result_reg == 1:
        return 2
    else:
        raise ValueError("Impossible measurement results!")

> Copyright (c) 2026 PsiQuantum